In [ ]:
# ======================================================================================
# DEFINITIVE CLEAN GAZE EXPERIMENT
# STEP 1 — FROZEN INPUT + FINAL GAZE TARGET CONSTRUCTION + BASE MODEL AUDIT
#
# RUN AFTER A FRESH KERNEL RESTART.
#
# IMPORTANT:
#     THIS SCRIPT DOES NOT TRAIN ANY MODEL.
#
# Frozen ECG input:
#     final_digitisation_v9/X_gaze_final.npy
#
# Gaze populations:
#     Consultant       = 9 consultants
#     All clinicians   = 9 consultants + 10 fellows + 1 resident = 20
#
# Gaze target construction:
#     1. Static AOI rows only
#     2. Exact source stimulus mapping
#     3. Exactly 12 diagnostic-grid AOIs
#     4. Per-reader 12-lead normalisation
#     5. Equal-weight averaging across readers
#
# Standard lead order:
#     I, II, III, aVR, aVL, aVF, V1, V2, V3, V4, V5, V6
#
# NO TRAINING IN THIS STEP.
# ======================================================================================


# ======================================================================================
# 0. IMPORTS
# ======================================================================================

import re
import json
import hashlib
import platform
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import tensorflow as tf


# ======================================================================================
# 1. REPRODUCIBILITY
# ======================================================================================

SEED = 2026

np.random.seed(SEED)
tf.random.set_seed(SEED)


print("=" * 100)
print("ENVIRONMENT")
print("=" * 100)

print("Python:", platform.python_version())
print("TensorFlow:", tf.__version__)
print("Working directory:", Path.cwd())


# ======================================================================================
# 2. FIXED EXPERIMENT DEFINITIONS
# ======================================================================================

ECG_ORDER = [
    "NSR",
    "AFib",
    "Hyperkalemia",
    "AFlutter",
    "VTach",
    "WPW",
    "VentPaced",
    "LBBB",
    "STEMI",
    "CompleteHeartBlock",
]


LEAD_ORDER = [
    "I",
    "II",
    "III",
    "aVR",
    "aVL",
    "aVF",
    "V1",
    "V2",
    "V3",
    "V4",
    "V5",
    "V6",
]


# Exact strings observed in ParentStimulus.
EXACT_STIMULUS_MAP = {

    "Normal sinus rhythm":
        "NSR",

    "Atrial fibrillation":
        "AFib",

    "hyperkalemia":
        "Hyperkalemia",

    "Atrial flutter":
        "AFlutter",

    "Ventricular tachycardia":
        "VTach",

    "Wolf Parkinson White syndrome":
        "WPW",

    "Ventricular paced rhythm":
        "VentPaced",

    "Left bundle branch block":
        "LBBB",

    "ST elevation MI":
        "STEMI",

    "Complete heart block":
        "CompleteHeartBlock",
}


# Diagnosis suffix used inside the Label column.
LABEL_SUFFIX = {

    "NSR":
        "NSR",

    "AFib":
        "AFib",

    "Hyperkalemia":
        "Hyperkalemia",

    "AFlutter":
        "AFlutter",

    "VTach":
        "VTach",

    "WPW":
        "WPW",

    "VentPaced":
        "VentPaced",

    "LBBB":
        "LBBB",

    "STEMI":
        "STEMI",

    "CompleteHeartBlock":
        "CompleteHeartBlock",
}


# Exact diagnostic-grid AOIs.
#
# IMPORTANT:
# Atrial flutter uses V1 short and V5 short in the source AOI definitions.
#
# Other AOIs such as:
#     II-1 / II-2 / ...
#     V1-1 / V5-1 / ...
#     Information
#     V3R / V4R / V7 / V8
# are deliberately excluded.
BASE_AOI_TO_LEAD = {

    "1":
        "I",

    "2":
        "II",

    "3":
        "III",

    "aVR":
        "aVR",

    "aVL":
        "aVL",

    "aVF":
        "aVF",

    "V1":
        "V1",

    "V2":
        "V2",

    "V3":
        "V3",

    "V4":
        "V4",

    "V5":
        "V5",

    "V6":
        "V6",

    "V1 short":
        "V1",

    "V5 short":
        "V5",
}


# ======================================================================================
# 3. PATHS
# ======================================================================================

OUT_DIR = Path(
    "definitive_clean_gaze"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# IMPORTANT:
# Do not fall back to an older X_gaze_final.npy.
V9_PATH = Path(
    "final_digitisation_v9/X_gaze_final.npy"
)


if not V9_PATH.exists():

    raise FileNotFoundError(
        "\nCould not find the frozen V9 dataset at:\n"
        f"    {V9_PATH}\n\n"
        "Do not substitute an earlier digitisation version."
    )


# Find gaze CSV by exact filename.
gaze_candidates = list(
    Path(".").glob(
        "**/Grid_Anonymized.csv"
    )
)


if not gaze_candidates:

    raise FileNotFoundError(
        "Could not find Grid_Anonymized.csv"
    )


gaze_candidates = sorted(
    gaze_candidates,
    key=lambda p: (
        len(p.parts),
        str(p)
    )
)


GAZE_CSV_PATH = gaze_candidates[0]


# Base model
base_candidates = (
    list(
        Path(".").glob(
            "**/12 leads.h5"
        )
    )
    +
    list(
        Path(".").glob(
            "**/12_leads.h5"
        )
    )
)


if not base_candidates:

    raise FileNotFoundError(
        "Could not find base model '12 leads.h5' or '12_leads.h5'."
    )


base_candidates = sorted(
    base_candidates,
    key=lambda p: (
        len(p.parts),
        str(p)
    )
)


BASE_MODEL_PATH = base_candidates[0]


print("\n" + "=" * 100)
print("FILES")
print("=" * 100)

print("Frozen V9 ECG input:", V9_PATH)
print("Eye-tracking CSV:   ", GAZE_CSV_PATH)
print("Base model:         ", BASE_MODEL_PATH)


# ======================================================================================
# 4. HASH HELPER
# ======================================================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            block = f.read(
                1024 * 1024
            )

            if not block:
                break

            h.update(block)

    return h.hexdigest()


V9_HASH = sha256_file(
    V9_PATH
)

GAZE_HASH = sha256_file(
    GAZE_CSV_PATH
)

BASE_HASH = sha256_file(
    BASE_MODEL_PATH
)


# ======================================================================================
# 5. LOAD FROZEN V9
# ======================================================================================

X_gaze = np.load(
    V9_PATH
).astype(
    np.float32
)


print("\n" + "=" * 100)
print("FROZEN V9 ECG INPUT")
print("=" * 100)

print("Shape:", X_gaze.shape)
print("Finite:", np.all(np.isfinite(X_gaze)))
print("SHA256:", V9_HASH)


if X_gaze.shape != (
    10,
    12,
    1000,
    1
):

    raise RuntimeError(
        f"Unexpected V9 shape: {X_gaze.shape}"
    )


if not np.all(
    np.isfinite(
        X_gaze
    )
):

    raise RuntimeError(
        "V9 contains non-finite values."
    )


# Each pseudo-waveform should have non-zero variance.
lead_sd = np.std(
    X_gaze[:, :, :, 0],
    axis=2
)


if np.any(
    lead_sd <= 1e-8
):

    bad = np.argwhere(
        lead_sd <= 1e-8
    )

    raise RuntimeError(
        f"At least one frozen pseudo-waveform has zero variance: {bad}"
    )


print(
    "\n✓ V9 shape and finite-value checks pass."
)


# ======================================================================================
# 6. LOAD RAW EYE-TRACKING CSV
# ======================================================================================

gaze = pd.read_csv(
    GAZE_CSV_PATH
)


required_columns = [
    "Respondent_Name",
    "Group",
    "Type",
    "Label",
    "ParentStimulus",
    "Time_spent_G",
]


missing_columns = [
    c
    for c in required_columns
    if c not in gaze.columns
]


if missing_columns:

    raise RuntimeError(
        f"Required columns missing: {missing_columns}"
    )


print("\n" + "=" * 100)
print("RAW EYE-TRACKING DATA")
print("=" * 100)

print("Shape:", gaze.shape)
print(
    "Unique respondents:",
    gaze["Respondent_Name"].nunique()
)
print(
    "Missing ParentStimulus rows:",
    gaze["ParentStimulus"].isna().sum()
)


# ======================================================================================
# 7. EXACT STIMULUS MAPPING
# ======================================================================================

actual_nonmissing_stimuli = set(
    gaze.loc[
        gaze["ParentStimulus"].notna(),
        "ParentStimulus"
    ].unique()
)


expected_stimuli = set(
    EXACT_STIMULUS_MAP.keys()
)


unexpected_stimuli = (
    actual_nonmissing_stimuli
    -
    expected_stimuli
)


missing_stimuli = (
    expected_stimuli
    -
    actual_nonmissing_stimuli
)


if unexpected_stimuli:

    raise RuntimeError(
        "\nUnexpected non-missing ParentStimulus values:\n"
        +
        "\n".join(
            sorted(unexpected_stimuli)
        )
    )


if missing_stimuli:

    raise RuntimeError(
        "\nExpected ParentStimulus values missing:\n"
        +
        "\n".join(
            sorted(missing_stimuli)
        )
    )


gaze["_ecg"] = gaze[
    "ParentStimulus"
].map(
    EXACT_STIMULUS_MAP
)


genuine_unmapped = gaze.loc[
    gaze["ParentStimulus"].notna(),
    "_ecg"
].isna().sum()


if genuine_unmapped != 0:

    raise RuntimeError(
        f"{genuine_unmapped} genuine stimulus rows remain unmapped."
    )


print("\n" + "=" * 100)
print("EXACT STIMULUS MAPPING")
print("=" * 100)


for source, ecg in EXACT_STIMULUS_MAP.items():

    n = int(
        (
            gaze["ParentStimulus"]
            ==
            source
        ).sum()
    )

    print(
        f"{source:35s} -> "
        f"{ecg:20s} "
        f"| rows={n}"
    )


print(
    "\n✓ All ten non-missing source stimuli mapped exactly."
)


# ======================================================================================
# 8. NORMALISE READER GROUPS
# ======================================================================================

gaze["_group_norm"] = (
    gaze["Group"]
    .astype(str)
    .str.strip()
    .str.lower()
)


reader_groups = (
    gaze[
        [
            "Respondent_Name",
            "_group_norm"
        ]
    ]
    .drop_duplicates()
)


# Every reader should belong to one group only.
groups_per_reader = (
    reader_groups
    .groupby("Respondent_Name")
    ["_group_norm"]
    .nunique()
)


if not (
    groups_per_reader == 1
).all():

    raise RuntimeError(
        "At least one Respondent_Name maps to multiple groups."
    )


print("\n" + "=" * 100)
print("READER GROUP AUDIT")
print("=" * 100)

print(
    reader_groups[
        "_group_norm"
    ]
    .value_counts()
    .to_string()
)


n_consultants = int(
    (
        reader_groups["_group_norm"]
        ==
        "consultant"
    ).sum()
)


n_fellows = int(
    (
        reader_groups["_group_norm"]
        ==
        "fellow"
    ).sum()
)


n_residents = int(
    (
        reader_groups["_group_norm"]
        ==
        "resident"
    ).sum()
)


print("\nConsultants:", n_consultants)
print("Fellows:    ", n_fellows)
print("Residents:  ", n_residents)


if n_consultants != 9:

    raise RuntimeError(
        f"Expected 9 consultants, found {n_consultants}."
    )


if n_fellows != 10:

    raise RuntimeError(
        f"Expected 10 fellows, found {n_fellows}."
    )


if n_residents != 1:

    raise RuntimeError(
        f"Expected 1 resident, found {n_residents}."
    )


print(
    "\n✓ 9 consultants + 10 fellows + 1 resident confirmed."
)


# ======================================================================================
# 9. FILTER STATIC AOIs
# ======================================================================================

static = gaze[
    gaze["Type"]
    .astype(str)
    .str.strip()
    .str.lower()
    ==
    "static aoi"
].copy()


print("\n" + "=" * 100)
print("STATIC AOI AUDIT")
print("=" * 100)

print(
    "Static AOI rows:",
    len(static)
)


if len(static) != 14742:

    raise RuntimeError(
        f"Expected 14,742 Static AOI rows, found {len(static)}."
    )


# Every Static AOI should have a mapped ECG.
if static["_ecg"].isna().any():

    bad = static.loc[
        static["_ecg"].isna(),
        [
            "ParentStimulus",
            "Label"
        ]
    ]

    raise RuntimeError(
        "\nAt least one Static AOI row has no ECG mapping:\n"
        +
        bad.head(20).to_string(index=False)
    )


print(
    "✓ 14,742 Static AOI rows confirmed."
)


# ======================================================================================
# 10. REMOVE ECG SUFFIX FROM AOI LABEL
# ======================================================================================

def extract_base_aoi(row):

    label = str(
        row["Label"]
    ).strip()

    ecg = row["_ecg"]

    suffix = LABEL_SUFFIX[
        ecg
    ]

    # Removes diagnosis suffix plus optional repeated "copy".
    #
    # Examples:
    #
    # V2 STEMI
    #     -> V2
    #
    # V1 short AFlutter
    #     -> V1 short
    #
    # II-2 CompleteHeartBlock copy
    #     -> II-2
    #
    # II-3 CompleteHeartBlock copy copy
    #     -> II-3

    pattern = (
        r"\s+"
        +
        re.escape(suffix)
        +
        r"(?:\s+copy)*$"
    )

    base = re.sub(
        pattern,
        "",
        label
    ).strip()

    return base


static["_base_aoi"] = static.apply(
    extract_base_aoi,
    axis=1
)


# ======================================================================================
# 11. MAP ONLY THE 12 DIAGNOSTIC LEADS
# ======================================================================================

static["_lead"] = static[
    "_base_aoi"
].map(
    BASE_AOI_TO_LEAD
)


lead_rows = static[
    static["_lead"].notna()
].copy()


lead_rows["_dwell"] = pd.to_numeric(
    lead_rows[
        "Time_spent_G"
    ],
    errors="coerce"
)


if lead_rows["_dwell"].isna().any():

    n_nan = int(
        lead_rows["_dwell"].isna().sum()
    )

    raise RuntimeError(
        f"{n_nan} selected diagnostic AOI rows have non-numeric Time_spent_G."
    )


if (
    lead_rows["_dwell"] < 0
).any():

    raise RuntimeError(
        "Negative dwell time detected."
    )


# ======================================================================================
# 12. STRUCTURAL AUDIT — 10 × 12 × 63
# ======================================================================================

audit_rows = []


for ecg in ECG_ORDER:

    temp = lead_rows[
        lead_rows["_ecg"]
        ==
        ecg
    ]


    lead_counts = (
        temp.groupby("_lead")
        .size()
        .reindex(LEAD_ORDER)
    )


    readers_per_lead = (
        temp.groupby("_lead")
        ["Respondent_Name"]
        .nunique()
        .reindex(LEAD_ORDER)
    )


    leads_present = set(
        temp["_lead"].unique()
    )


    audit_rows.append(
        {
            "ECG":
                ecg,

            "n_selected_rows":
                len(temp),

            "n_leads":
                len(leads_present),

            "min_rows_per_lead":
                lead_counts.min(),

            "max_rows_per_lead":
                lead_counts.max(),

            "min_readers_per_lead":
                readers_per_lead.min(),

            "max_readers_per_lead":
                readers_per_lead.max(),

            "missing_leads":
                (
                    ", ".join(
                        sorted(
                            set(LEAD_ORDER)
                            -
                            leads_present
                        )
                    )
                    or
                    "NONE"
                ),
        }
    )


structure_df = pd.DataFrame(
    audit_rows
)


print("\n" + "=" * 100)
print("FINAL 12-LEAD STRUCTURAL AUDIT")
print("=" * 100)

print(
    structure_df.to_string(
        index=False
    )
)


if not (
    structure_df["n_selected_rows"]
    ==
    12 * 63
).all():

    raise RuntimeError(
        "At least one ECG does not contain exactly 12 × 63 selected rows."
    )


if not (
    structure_df["n_leads"]
    ==
    12
).all():

    raise RuntimeError(
        "At least one ECG does not contain 12 selected diagnostic leads."
    )


if not (
    structure_df["min_rows_per_lead"]
    ==
    63
).all():

    raise RuntimeError(
        "At least one ECG/lead has fewer than 63 rows."
    )


if not (
    structure_df["max_rows_per_lead"]
    ==
    63
).all():

    raise RuntimeError(
        "At least one ECG/lead has more than 63 rows."
    )


if not (
    structure_df["min_readers_per_lead"]
    ==
    63
).all():

    raise RuntimeError(
        "At least one ECG/lead has fewer than 63 unique readers."
    )


if not (
    structure_df["max_readers_per_lead"]
    ==
    63
).all():

    raise RuntimeError(
        "At least one ECG/lead has more than 63 unique readers."
    )


print(
    "\n✓ 10 ECGs × 12 diagnostic leads × 63 readers confirmed."
)


# ======================================================================================
# 13. SHOW SPECIAL ATRIAL-FLUTTER MAPPING
# ======================================================================================

aflutter_mapping = (
    lead_rows[
        lead_rows["_ecg"]
        ==
        "AFlutter"
    ][
        [
            "_base_aoi",
            "_lead"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "_lead"
    )
)


print("\n" + "=" * 100)
print("ATRIAL FLUTTER DIAGNOSTIC AOIs")
print("=" * 100)

print(
    aflutter_mapping.to_string(
        index=False
    )
)


# Must explicitly use source special labels.
af_source = set(
    aflutter_mapping["_base_aoi"]
)


if "V1 short" not in af_source:

    raise RuntimeError(
        "AFlutter V1 short AOI was not selected."
    )


if "V5 short" not in af_source:

    raise RuntimeError(
        "AFlutter V5 short AOI was not selected."
    )


print(
    "\n✓ AFlutter V1 short -> V1 and V5 short -> V5 confirmed."
)


# ======================================================================================
# 14. EACH READER × ECG MUST HAVE EXACTLY 12 SELECTED AOIs
# ======================================================================================

reader_ecg_counts = (
    lead_rows
    .groupby(
        [
            "Respondent_Name",
            "_ecg"
        ]
    )
    .size()
)


if not (
    reader_ecg_counts
    ==
    12
).all():

    bad = reader_ecg_counts[
        reader_ecg_counts
        !=
        12
    ]

    raise RuntimeError(
        "\nAt least one reader × ECG does not have exactly 12 selected AOIs:\n"
        +
        bad.head(30).to_string()
    )


n_reader_ecg_pairs = len(
    reader_ecg_counts
)


expected_pairs = (
    63
    *
    10
)


if n_reader_ecg_pairs != expected_pairs:

    raise RuntimeError(
        f"Expected {expected_pairs} reader×ECG pairs, "
        f"found {n_reader_ecg_pairs}."
    )


print("\n" + "=" * 100)
print("READER × ECG COMPLETENESS")
print("=" * 100)

print(
    f"Reader × ECG pairs: {n_reader_ecg_pairs}"
)

print(
    "Selected diagnostic AOIs per pair:",
    reader_ecg_counts.min(),
    "to",
    reader_ecg_counts.max()
)

print(
    "\n✓ Every one of the 63 readers has exactly 12 selected leads for all 10 ECGs."
)


# ======================================================================================
# 15. CREATE ONE RAW 12-LEAD VECTOR PER READER × ECG
# ======================================================================================

# Because structural audit proved uniqueness, pivot is now safe.
reader_matrix = (
    lead_rows
    .pivot(
        index=[
            "Respondent_Name",
            "_group_norm",
            "_ecg"
        ],
        columns="_lead",
        values="_dwell"
    )
    .reindex(
        columns=LEAD_ORDER
    )
)


if reader_matrix.isna().any().any():

    raise RuntimeError(
        "Missing values appeared in reader × ECG × lead pivot."
    )


# Preserve raw totals before normalisation.
reader_matrix[
    "_raw_total"
] = reader_matrix[
    LEAD_ORDER
].sum(
    axis=1
)


if (
    reader_matrix["_raw_total"]
    <=
    0
).any():

    bad = reader_matrix[
        reader_matrix["_raw_total"]
        <=
        0
    ]

    raise RuntimeError(
        "\nAt least one reader × ECG has zero total dwell time:\n"
        +
        bad.head(20).to_string()
    )


# ======================================================================================
# 16. PER-READER NORMALISATION
# ======================================================================================

reader_norm = reader_matrix.copy()


reader_norm[
    LEAD_ORDER
] = (
    reader_norm[
        LEAD_ORDER
    ]
    .div(
        reader_norm["_raw_total"],
        axis=0
    )
)


reader_norm_sums = (
    reader_norm[
        LEAD_ORDER
    ].sum(
        axis=1
    )
)


if not np.allclose(
    reader_norm_sums.values,
    1.0,
    atol=1e-10
):

    raise RuntimeError(
        "Per-reader gaze normalisation failed."
    )


print("\n" + "=" * 100)
print("PER-READER NORMALISATION")
print("=" * 100)

print(
    "Min normalised row sum:",
    reader_norm_sums.min()
)

print(
    "Max normalised row sum:",
    reader_norm_sums.max()
)

print(
    "\n✓ Every reader × ECG vector independently sums to 1."
)


# ======================================================================================
# 17. DEFINE THE TWO CLEAN POPULATIONS
# ======================================================================================

consultant_reader_ids = sorted(
    reader_groups.loc[
        reader_groups["_group_norm"]
        ==
        "consultant",
        "Respondent_Name"
    ].unique()
)


all_clinician_reader_ids = sorted(
    reader_groups.loc[
        reader_groups["_group_norm"].isin(
            [
                "consultant",
                "fellow",
                "resident"
            ]
        ),
        "Respondent_Name"
    ].unique()
)


if len(
    consultant_reader_ids
) != 9:

    raise RuntimeError(
        "Consultant reader population is not exactly 9."
    )


if len(
    all_clinician_reader_ids
) != 20:

    raise RuntimeError(
        "All-clinician reader population is not exactly 20."
    )


# ======================================================================================
# 18. VERIFY ALL SELECTED CLINICIANS HAVE ALL 10 ECGs
# ======================================================================================

def audit_population_completeness(
    reader_ids,
    population_name
):

    temp = (
        reader_norm
        .reset_index()
    )


    temp = temp[
        temp[
            "Respondent_Name"
        ].isin(
            reader_ids
        )
    ]


    counts = (
        temp
        .groupby(
            "Respondent_Name"
        )["_ecg"]
        .nunique()
    )


    if not (
        counts == 10
    ).all():

        bad = counts[
            counts != 10
        ]

        raise RuntimeError(
            f"{population_name}: incomplete ECG coverage:\n"
            +
            bad.to_string()
        )


    return counts


consultant_complete = audit_population_completeness(
    consultant_reader_ids,
    "Consultant"
)


all_complete = audit_population_completeness(
    all_clinician_reader_ids,
    "All clinicians"
)


print("\n" + "=" * 100)
print("CLINICAL POPULATION COMPLETENESS")
print("=" * 100)

print(
    "Consultants:",
    len(consultant_complete),
    "readers × 10 ECGs"
)

print(
    "All clinicians:",
    len(all_complete),
    "readers × 10 ECGs"
)

print(
    "\n✓ All 9 consultants contributed to all ten ECGs."
)

print(
    "✓ All 20 clinicians contributed to all ten ECGs."
)


# ======================================================================================
# 19. BUILD EQUAL-READER-WEIGHTED TARGETS
# ======================================================================================

reader_norm_reset = (
    reader_norm
    .reset_index()
)


def build_population_target(
    reader_ids,
    population_name
):

    temp = reader_norm_reset[
        reader_norm_reset[
            "Respondent_Name"
        ].isin(
            reader_ids
        )
    ].copy()


    targets = []


    for ecg in ECG_ORDER:

        ecg_temp = temp[
            temp["_ecg"]
            ==
            ecg
        ].copy()


        # There must be exactly one normalised vector per reader.
        if len(ecg_temp) != len(reader_ids):

            raise RuntimeError(
                f"{population_name} / {ecg}: "
                f"expected {len(reader_ids)} readers, found {len(ecg_temp)}."
            )


        if ecg_temp[
            "Respondent_Name"
        ].nunique() != len(reader_ids):

            raise RuntimeError(
                f"{population_name} / {ecg}: duplicate or missing readers."
            )


        matrix = ecg_temp[
            LEAD_ORDER
        ].to_numpy(
            dtype=np.float64
        )


        # Equal reader weighting.
        target = matrix.mean(
            axis=0
        )


        # Defensive renormalisation against floating-point drift.
        target = (
            target
            /
            target.sum()
        )


        targets.append(
            target
        )


    targets = np.stack(
        targets,
        axis=0
    )


    return targets


Y_consultant = build_population_target(
    consultant_reader_ids,
    "Consultant"
).astype(
    np.float32
)


Y_all_clinicians = build_population_target(
    all_clinician_reader_ids,
    "All clinicians"
).astype(
    np.float32
)


if Y_consultant.shape != (
    10,
    12
):

    raise RuntimeError(
        f"Unexpected consultant target shape: {Y_consultant.shape}"
    )


if Y_all_clinicians.shape != (
    10,
    12
):

    raise RuntimeError(
        f"Unexpected all-clinician target shape: {Y_all_clinicians.shape}"
    )


if not np.allclose(
    Y_consultant.sum(axis=1),
    1.0,
    atol=1e-6
):

    raise RuntimeError(
        "Consultant population targets do not sum to one."
    )


if not np.allclose(
    Y_all_clinicians.sum(axis=1),
    1.0,
    atol=1e-6
):

    raise RuntimeError(
        "All-clinician population targets do not sum to one."
    )


# ======================================================================================
# 20. DISPLAY FINAL TARGETS
# ======================================================================================

consultant_df = pd.DataFrame(
    Y_consultant,
    index=ECG_ORDER,
    columns=LEAD_ORDER
)


all_df = pd.DataFrame(
    Y_all_clinicians,
    index=ECG_ORDER,
    columns=LEAD_ORDER
)


print("\n" + "=" * 100)
print("FINAL CONSULTANT GAZE TARGET — 9 READERS")
print("=" * 100)

print(
    consultant_df
    .round(4)
    .to_string()
)


print("\n" + "=" * 100)
print("FINAL ALL-CLINICIAN GAZE TARGET — 20 READERS")
print("=" * 100)

print(
    all_df
    .round(4)
    .to_string()
)


# ======================================================================================
# 21. CONSULTANT vs ALL-CLINICIAN TARGET DIFFERENCE
# ======================================================================================

def cosine_similarity(
    a,
    b
):

    denominator = (
        np.linalg.norm(a)
        *
        np.linalg.norm(b)
    )

    if denominator <= 0:
        raise RuntimeError(
            "Zero norm in cosine similarity."
        )

    return float(
        np.dot(a, b)
        /
        denominator
    )


comparison_rows = []


for i, ecg in enumerate(
    ECG_ORDER
):

    consultant = Y_consultant[i]
    all_clin = Y_all_clinicians[i]


    comparison_rows.append(
        {
            "ECG":
                ecg,

            "cosine_similarity":
                cosine_similarity(
                    consultant,
                    all_clin
                ),

            "L1_distance":
                float(
                    np.sum(
                        np.abs(
                            consultant
                            -
                            all_clin
                        )
                    )
                ),

            "max_abs_lead_difference":
                float(
                    np.max(
                        np.abs(
                            consultant
                            -
                            all_clin
                        )
                    )
                ),
        }
    )


comparison_df = pd.DataFrame(
    comparison_rows
)


print("\n" + "=" * 100)
print("CONSULTANT vs ALL-CLINICIAN TARGET DIFFERENCE")
print("=" * 100)

print(
    comparison_df
    .round(6)
    .to_string(
        index=False
    )
)


# ======================================================================================
# 22. SAVE FINAL TARGETS
# ======================================================================================

TARGET_NPZ = (
    OUT_DIR
    /
    "clean_gaze_targets.npz"
)


np.savez_compressed(

    TARGET_NPZ,

    consultant=
        Y_consultant,

    all_clinicians=
        Y_all_clinicians,

    ecg_order=
        np.asarray(
            ECG_ORDER
        ),

    lead_order=
        np.asarray(
            LEAD_ORDER
        ),
)


# Long population-target table
target_long_rows = []


for ei, ecg in enumerate(
    ECG_ORDER
):

    for li, lead in enumerate(
        LEAD_ORDER
    ):

        target_long_rows.append(
            {
                "ECG":
                    ecg,

                "lead":
                    lead,

                "consultant":
                    float(
                        Y_consultant[
                            ei,
                            li
                        ]
                    ),

                "all_clinicians":
                    float(
                        Y_all_clinicians[
                            ei,
                            li
                        ]
                    ),
            }
        )


pd.DataFrame(
    target_long_rows
).to_csv(

    OUT_DIR
    /
    "clean_gaze_targets_long.csv",

    index=False
)


# Per-reader normalised vectors for audit/reproducibility
reader_norm_export = (
    reader_norm_reset[
        [
            "Respondent_Name",
            "_group_norm",
            "_ecg",
            "_raw_total",
            *LEAD_ORDER
        ]
    ]
    .rename(
        columns={
            "_group_norm":
                "group",

            "_ecg":
                "ECG",

            "_raw_total":
                "raw_total_dwell"
        }
    )
)


reader_norm_export.to_csv(

    OUT_DIR
    /
    "per_reader_normalised_gaze.csv",

    index=False
)


structure_df.to_csv(

    OUT_DIR
    /
    "structural_audit.csv",

    index=False
)


comparison_df.to_csv(

    OUT_DIR
    /
    "population_target_comparison.csv",

    index=False
)


# ======================================================================================
# 23. HASH SAVED TARGET FILE
# ======================================================================================

TARGET_HASH = sha256_file(
    TARGET_NPZ
)


print("\nSaved target file:")
print(TARGET_NPZ)

print("\nTarget SHA256:")
print(TARGET_HASH)


# ======================================================================================
# 24. LOAD BASE MODEL — NO TRAINING
# ======================================================================================

print("\n" + "=" * 100)
print("BASE MODEL AUDIT")
print("=" * 100)


base_model = tf.keras.models.load_model(
    BASE_MODEL_PATH,
    compile=False
)


print(
    "Model name:",
    base_model.name
)

print(
    "Input shape:",
    base_model.input_shape
)

print(
    "Output shape:",
    base_model.output_shape
)

print(
    "Number of layers:",
    len(base_model.layers)
)


# Required trainable layer
try:

    conv10 = base_model.get_layer(
        "conv2d_10"
    )

except Exception as exc:

    raise RuntimeError(
        "Base model does not contain conv2d_10."
    ) from exc


print(
    "\nconv2d_10 output shape:",
    conv10.output.shape
)


# Representation layer
try:

    add3 = base_model.get_layer(
        "add_3"
    )

except Exception as exc:

    raise RuntimeError(
        "Base model does not contain add_3."
    ) from exc


print(
    "add_3 output shape:",
    add3.output.shape
)


final_layer = base_model.layers[-1]


print(
    "\nFinal layer:",
    final_layer.name
)


if hasattr(
    final_layer,
    "activation"
):

    print(
        "Final activation:",
        final_layer.activation.__name__
    )


print(
    "\nBase model SHA256:",
    BASE_HASH
)


# ======================================================================================
# 25. FORWARD PASS — STILL NO TRAINING
# ======================================================================================

print("\n" + "=" * 100)
print("FORWARD-PASS CHECK")
print("=" * 100)


test_output = base_model(
    X_gaze[
        0:1
    ],
    training=False
)


if isinstance(
    test_output,
    (
        list,
        tuple
    )
):

    print(
        "Number of outputs:",
        len(test_output)
    )

    for i, out in enumerate(
        test_output
    ):

        print(
            f"Output {i}:",
            tuple(out.shape)
        )

else:

    print(
        "Output shape:",
        tuple(test_output.shape)
    )


if not np.all(
    np.isfinite(
        np.asarray(
            test_output
        )
    )
):

    raise RuntimeError(
        "Base-model forward pass produced non-finite values."
    )


print(
    "\n✓ Base-model forward pass successful."
)


# ======================================================================================
# 26. SAVE DEFINITIVE STEP-1 MANIFEST
# ======================================================================================

manifest = {

    "created":
        datetime.now().isoformat(),

    "experiment":
        "Definitive clean gaze-aligned ECG experiment",

    "stage":
        "Step 1: frozen inputs and gaze target construction",

    "random_seed":
        SEED,

    "frozen_ecg_input":
        str(V9_PATH),

    "frozen_ecg_sha256":
        V9_HASH,

    "frozen_ecg_shape":
        list(X_gaze.shape),

    "eye_tracking_csv":
        str(GAZE_CSV_PATH),

    "eye_tracking_csv_sha256":
        GAZE_HASH,

    "base_model":
        str(BASE_MODEL_PATH),

    "base_model_sha256":
        BASE_HASH,

    "clean_gaze_targets":
        str(TARGET_NPZ),

    "clean_gaze_targets_sha256":
        TARGET_HASH,

    "ecg_order":
        ECG_ORDER,

    "lead_order":
        LEAD_ORDER,

    "consultant_reader_count":
        9,

    "all_clinician_reader_count":
        20,

    "all_clinician_composition": {
        "consultant":
            9,

        "fellow":
            10,

        "resident":
            1,
    },

    "stimulus_mapping":
        EXACT_STIMULUS_MAP,

    "diagnostic_aoi_mapping":
        BASE_AOI_TO_LEAD,

    "gaze_target_method": (
        "For each reader and ECG, dwell times within the 12 selected "
        "diagnostic-grid AOIs were arranged in fixed standard lead order "
        "and independently normalised to sum to one. Population targets "
        "were then calculated as the arithmetic mean of these normalised "
        "reader vectors, giving every reader equal weight."
    ),

    "static_aoi_rows":
        int(len(static)),

    "selected_diagnostic_rows":
        int(len(lead_rows)),

    "training_performed":
        False,
}


with open(

    OUT_DIR
    /
    "step1_manifest.json",

    "w"

) as f:

    json.dump(
        manifest,
        f,
        indent=2
    )


# ======================================================================================
# 27. FINAL REPORT
# ======================================================================================

print("\n\n" + "=" * 100)
print("STEP 1 COMPLETE — NO TRAINING PERFORMED")
print("=" * 100)


print(
    """
If this cell reaches here without an exception, the clean gaze targets
have been constructed from a fresh kernel using a single immutable pipeline.

PLEASE PASTE BACK THESE FOUR OUTPUT SECTIONS:

1. FINAL 12-LEAD STRUCTURAL AUDIT

2. CLINICAL POPULATION COMPLETENESS

3. CONSULTANT vs ALL-CLINICIAN TARGET DIFFERENCE

4. BASE MODEL AUDIT + FORWARD-PASS CHECK

Do not train anything yet.
"""
)

In [ ]:
# ======================================================================================
# DIAGNOSTIC — ZERO-DWELL READER × ECG PAIRS
#
# Determine whether zero 12-lead dwell occurs among:
#   - any of the 63 readers
#   - specifically the 20 clinicians used in the clean experiment
#
# Also inspect where CCU 1 looked during VTach.
# ======================================================================================


# reader_matrix already exists from the clean script
zero_pairs = (
    reader_matrix[
        reader_matrix["_raw_total"] <= 0
    ]
    .reset_index()
)


print("=" * 100)
print("ALL ZERO-DWELL READER × ECG PAIRS")
print("=" * 100)

print(
    "Number of zero-total pairs:",
    len(zero_pairs)
)

if len(zero_pairs) > 0:

    print(
        zero_pairs[
            [
                "Respondent_Name",
                "_group_norm",
                "_ecg",
                "_raw_total",
            ]
        ].to_string(
            index=False
        )
    )


# ======================================================================================
# Check ONLY populations used in the definitive experiment
# ======================================================================================

included_groups = {
    "consultant",
    "fellow",
    "resident",
}


zero_included = zero_pairs[
    zero_pairs["_group_norm"].isin(
        included_groups
    )
].copy()


print("\n" + "=" * 100)
print("ZERO-DWELL PAIRS AMONG THE 20 INCLUDED CLINICIANS")
print("=" * 100)

print(
    "Number:",
    len(zero_included)
)

if len(zero_included) > 0:

    print(
        zero_included[
            [
                "Respondent_Name",
                "_group_norm",
                "_ecg",
                "_raw_total",
            ]
        ].to_string(
            index=False
        )
    )

else:

    print(
        "NONE"
    )


# ======================================================================================
# Inspect CCU 1 / VTach across ALL Static AOIs
# ======================================================================================

inspect = static[
    (
        static["Respondent_Name"] == "CCU 1"
    )
    &
    (
        static["_ecg"] == "VTach"
    )
][
    [
        "_base_aoi",
        "_lead",
        "Time_spent_G",
    ]
].copy()


inspect["Time_spent_G"] = pd.to_numeric(
    inspect["Time_spent_G"],
    errors="coerce"
).fillna(0.0)


inspect = inspect.sort_values(
    "Time_spent_G",
    ascending=False
)


print("\n" + "=" * 100)
print("CCU 1 — VTACH: ALL STATIC AOIs")
print("=" * 100)

print(
    inspect.to_string(
        index=False
    )
)


print(
    "\nTotal dwell in selected 12 leads:",
    inspect.loc[
        inspect["_lead"].notna(),
        "Time_spent_G"
    ].sum()
)


print(
    "Total dwell in excluded AOIs:",
    inspect.loc[
        inspect["_lead"].isna(),
        "Time_spent_G"
    ].sum()
)

In [ ]:
# ======================================================================================
# DIAGNOSTIC — HOW MUCH CLINICIAN GAZE ARE WE DISCARDING?
#
# No targets are built and no model is trained.
# ======================================================================================

included_groups = {
    "consultant",
    "fellow",
    "resident",
}

clinical_static = static[
    static["_group_norm"].isin(included_groups)
].copy()


# Numeric dwell
clinical_static["_dwell_numeric"] = pd.to_numeric(
    clinical_static["Time_spent_G"],
    errors="coerce"
).fillna(0.0)


# Whether current strict 12-grid mapping retains this AOI
clinical_static["_currently_selected"] = (
    clinical_static["_lead"].notna()
)


# ======================================================================================
# 1. Per reader × ECG retained-vs-excluded gaze
# ======================================================================================

coverage = (
    clinical_static
    .groupby(
        [
            "Respondent_Name",
            "_group_norm",
            "_ecg"
        ]
    )
    .apply(
        lambda df: pd.Series(
            {
                "total_static_dwell":
                    df["_dwell_numeric"].sum(),

                "selected_12_dwell":
                    df.loc[
                        df["_currently_selected"],
                        "_dwell_numeric"
                    ].sum(),

                "excluded_static_dwell":
                    df.loc[
                        ~df["_currently_selected"],
                        "_dwell_numeric"
                    ].sum(),
            }
        ),
        include_groups=False
    )
    .reset_index()
)


coverage["selected_fraction"] = np.where(
    coverage["total_static_dwell"] > 0,
    coverage["selected_12_dwell"] /
    coverage["total_static_dwell"],
    np.nan
)


coverage["excluded_fraction"] = np.where(
    coverage["total_static_dwell"] > 0,
    coverage["excluded_static_dwell"] /
    coverage["total_static_dwell"],
    np.nan
)


print("=" * 100)
print("CLINICAL GAZE RETAINED BY CURRENT 12-AOI DEFINITION")
print("=" * 100)

print(
    coverage[
        [
            "selected_fraction",
            "excluded_fraction"
        ]
    ]
    .describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90
        ]
    )
    .round(4)
    .to_string()
)


# ======================================================================================
# 2. By ECG
# ======================================================================================

by_ecg = (
    coverage
    .groupby("_ecg")
    .agg(
        n_readers=(
            "Respondent_Name",
            "nunique"
        ),
        median_selected_fraction=(
            "selected_fraction",
            "median"
        ),
        mean_selected_fraction=(
            "selected_fraction",
            "mean"
        ),
        min_selected_fraction=(
            "selected_fraction",
            "min"
        ),
        median_excluded_fraction=(
            "excluded_fraction",
            "median"
        ),
        max_excluded_fraction=(
            "excluded_fraction",
            "max"
        ),
    )
    .reindex(ECG_ORDER)
    .reset_index()
)


print("\n" + "=" * 100)
print("CURRENT GAZE COVERAGE BY ECG — 20 INCLUDED CLINICIANS")
print("=" * 100)

print(
    by_ecg
    .round(4)
    .to_string(index=False)
)


# ======================================================================================
# 3. Worst clinician × ECG cases
# ======================================================================================

print("\n" + "=" * 100)
print("20 LOWEST RETAINED-GAZE CLINICIAN × ECG PAIRS")
print("=" * 100)

print(
    coverage
    .sort_values("selected_fraction")
    [
        [
            "Respondent_Name",
            "_group_norm",
            "_ecg",
            "total_static_dwell",
            "selected_12_dwell",
            "excluded_static_dwell",
            "selected_fraction",
            "excluded_fraction",
        ]
    ]
    .head(20)
    .round(4)
    .to_string(index=False)
)


# ======================================================================================
# 4. Which excluded AOIs receive the most gaze?
# ======================================================================================

excluded = clinical_static[
    ~clinical_static["_currently_selected"]
].copy()


excluded_summary = (
    excluded
    .groupby(
        [
            "_ecg",
            "_base_aoi"
        ],
        as_index=False
    )
    .agg(
        total_dwell=(
            "_dwell_numeric",
            "sum"
        ),
        readers_with_positive_dwell=(
            "_dwell_numeric",
            lambda x: int(
                np.sum(
                    np.asarray(x) > 0
                )
            )
        )
    )
)


excluded_summary = excluded_summary.sort_values(
    [
        "_ecg",
        "total_dwell"
    ],
    ascending=[
        True,
        False
    ]
)


print("\n" + "=" * 100)
print("TOP EXCLUDED AOIs BY ECG")
print("=" * 100)

for ecg in ECG_ORDER:

    temp = excluded_summary[
        excluded_summary["_ecg"] == ecg
    ].head(12)

    print("\n---", ecg, "---")

    if len(temp) == 0:
        print("NONE")
    else:
        print(
            temp.round(1).to_string(index=False)
        )

In [ ]:
# ======================================================================================
# DEFINITIVE CLEAN GAZE EXPERIMENT
# STEP 1 — CONTINUATION AFTER AOI-COVERAGE AUDIT
#
# IMPORTANT METHODOLOGICAL DEFINITION
# -----------------------------------
# Primary gaze target = gaze CONDITIONAL ON THE 12 MODEL-VISIBLE SHORT-LEAD AOIs.
#
# Excluded:
#     - long rhythm-strip grid cells (II-*, V1-*, V5-*, etc.)
#     - Information
#     - non-standard leads such as V3R, V4R, V7, V8
#
# These regions are not represented as separate inputs in X_gaze.
#
# No training occurs here.
# ======================================================================================

import json
import hashlib
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import tensorflow as tf


# ======================================================================================
# 1. PRESPECIFIED CLINICAL POPULATIONS
# ======================================================================================

CONSULTANT_GROUPS = {
    "consultant"
}

ALL_CLINICIAN_GROUPS = {
    "consultant",
    "fellow",
    "resident",
}


consultant_reader_ids = sorted(
    reader_groups.loc[
        reader_groups["_group_norm"].isin(
            CONSULTANT_GROUPS
        ),
        "Respondent_Name"
    ].unique()
)


all_clinician_reader_ids = sorted(
    reader_groups.loc[
        reader_groups["_group_norm"].isin(
            ALL_CLINICIAN_GROUPS
        ),
        "Respondent_Name"
    ].unique()
)


if len(consultant_reader_ids) != 9:
    raise RuntimeError(
        f"Expected 9 consultants, found {len(consultant_reader_ids)}."
    )


if len(all_clinician_reader_ids) != 20:
    raise RuntimeError(
        f"Expected 20 clinicians, found {len(all_clinician_reader_ids)}."
    )


print("=" * 100)
print("PRESPECIFIED CLINICAL POPULATIONS")
print("=" * 100)

print("Consultants:    ", len(consultant_reader_ids))
print("All clinicians:", len(all_clinician_reader_ids))

print(
    "\n✓ 9-consultant and 20-clinician populations confirmed."
)


# ======================================================================================
# 2. RESTRICT READER MATRIX BEFORE NORMALISATION
# ======================================================================================

clinical_matrix = (
    reader_matrix
    .reset_index()
)


clinical_matrix = clinical_matrix[
    clinical_matrix[
        "Respondent_Name"
    ].isin(
        all_clinician_reader_ids
    )
].copy()


if len(clinical_matrix) != 20 * 10:

    raise RuntimeError(
        f"Expected 200 clinician×ECG vectors, found {len(clinical_matrix)}."
    )


if clinical_matrix[
    "Respondent_Name"
].nunique() != 20:

    raise RuntimeError(
        "Expected 20 unique clinicians."
    )


# ======================================================================================
# 3. VERIFY COMPLETE ECG COVERAGE
# ======================================================================================

clinical_completeness = (
    clinical_matrix
    .groupby(
        [
            "Respondent_Name",
            "_group_norm"
        ]
    )["_ecg"]
    .nunique()
    .reset_index(
        name="n_ECGs"
    )
)


print("\n" + "=" * 100)
print("CLINICAL POPULATION COMPLETENESS")
print("=" * 100)

print(
    clinical_completeness.to_string(
        index=False
    )
)


if not (
    clinical_completeness["n_ECGs"]
    ==
    10
).all():

    raise RuntimeError(
        "At least one included clinician does not have all ten ECGs."
    )


# ======================================================================================
# 4. VERIFY POSITIVE MODEL-VISIBLE DWELL
# ======================================================================================

zero_clinical = clinical_matrix[
    clinical_matrix["_raw_total"]
    <=
    0
]


if len(zero_clinical) > 0:

    raise RuntimeError(
        "\nAn INCLUDED clinician has zero model-visible 12-lead dwell:\n"
        +
        zero_clinical[
            [
                "Respondent_Name",
                "_group_norm",
                "_ecg",
                "_raw_total"
            ]
        ].to_string(index=False)
    )


print(
    "\n✓ All 200 included clinician×ECG vectors have positive "
    "dwell within the model-visible 12 leads."
)


# ======================================================================================
# 5. NORMALISE EACH READER × ECG INDEPENDENTLY
# ======================================================================================

clinical_norm = clinical_matrix.copy()


clinical_norm[
    LEAD_ORDER
] = clinical_norm[
    LEAD_ORDER
].div(
    clinical_norm["_raw_total"],
    axis=0
)


normalised_sums = clinical_norm[
    LEAD_ORDER
].sum(
    axis=1
)


if not np.allclose(
    normalised_sums,
    1.0,
    atol=1e-10
):

    raise RuntimeError(
        "Per-reader normalisation failed."
    )


print("\n" + "=" * 100)
print("PER-READER MODEL-VISIBLE GAZE NORMALISATION")
print("=" * 100)

print(
    "Minimum vector sum:",
    normalised_sums.min()
)

print(
    "Maximum vector sum:",
    normalised_sums.max()
)

print(
    "\n✓ Each clinician×ECG 12-lead gaze vector independently sums to 1."
)


# ======================================================================================
# 6. RECORD HOW MUCH WHOLE-IMAGE STATIC GAZE IS REPRESENTED
#
# This is an audit/limitation metric, not used to create the target.
# ======================================================================================

clinical_static = static[
    static["_group_norm"].isin(
        ALL_CLINICIAN_GROUPS
    )
].copy()


clinical_static[
    "_dwell_numeric"
] = pd.to_numeric(
    clinical_static[
        "Time_spent_G"
    ],
    errors="coerce"
).fillna(
    0.0
)


total_static = (
    clinical_static
    .groupby(
        [
            "Respondent_Name",
            "_group_norm",
            "_ecg"
        ]
    )["_dwell_numeric"]
    .sum()
    .rename(
        "whole_static_dwell"
    )
)


selected_static = (
    clinical_static[
        clinical_static["_lead"].notna()
    ]
    .groupby(
        [
            "Respondent_Name",
            "_group_norm",
            "_ecg"
        ]
    )["_dwell_numeric"]
    .sum()
    .rename(
        "model_visible_dwell"
    )
)


coverage_df = pd.concat(
    [
        total_static,
        selected_static
    ],
    axis=1
).fillna(
    0
).reset_index()


coverage_df[
    "excluded_dwell"
] = (
    coverage_df[
        "whole_static_dwell"
    ]
    -
    coverage_df[
        "model_visible_dwell"
    ]
)


coverage_df[
    "model_visible_fraction"
] = (
    coverage_df[
        "model_visible_dwell"
    ]
    /
    coverage_df[
        "whole_static_dwell"
    ]
)


coverage_df[
    "excluded_fraction"
] = (
    coverage_df[
        "excluded_dwell"
    ]
    /
    coverage_df[
        "whole_static_dwell"
    ]
)


print("\n" + "=" * 100)
print("MODEL-VISIBLE GAZE COVERAGE — AUDIT ONLY")
print("=" * 100)

print(
    coverage_df[
        [
            "model_visible_fraction",
            "excluded_fraction"
        ]
    ]
    .describe()
    .round(4)
    .to_string()
)


print(
    "\nMean model-visible fraction:",
    round(
        coverage_df[
            "model_visible_fraction"
        ].mean(),
        4
    )
)

print(
    "Median model-visible fraction:",
    round(
        coverage_df[
            "model_visible_fraction"
        ].median(),
        4
    )
)


# ======================================================================================
# 7. FUNCTION TO BUILD EQUAL-READER TARGET
# ======================================================================================

def build_target(
    reader_ids,
    population_name
):

    pop = clinical_norm[
        clinical_norm[
            "Respondent_Name"
        ].isin(
            reader_ids
        )
    ].copy()


    targets = []


    for ecg in ECG_ORDER:

        temp = pop[
            pop["_ecg"]
            ==
            ecg
        ].copy()


        if len(temp) != len(reader_ids):

            raise RuntimeError(
                f"{population_name}/{ecg}: "
                f"expected {len(reader_ids)} vectors, found {len(temp)}."
            )


        if temp[
            "Respondent_Name"
        ].nunique() != len(reader_ids):

            raise RuntimeError(
                f"{population_name}/{ecg}: duplicate/missing reader."
            )


        matrix = temp[
            LEAD_ORDER
        ].to_numpy(
            dtype=np.float64
        )


        # Equal reader weighting
        target = matrix.mean(
            axis=0
        )


        # Defensive numerical renormalisation
        target /= target.sum()


        targets.append(
            target
        )


    return np.stack(
        targets,
        axis=0
    ).astype(
        np.float32
    )


# ======================================================================================
# 8. BUILD BOTH DEFINITIVE TARGETS
# ======================================================================================

Y_consultant = build_target(
    consultant_reader_ids,
    "Consultant"
)


Y_all_clinicians = build_target(
    all_clinician_reader_ids,
    "All clinicians"
)


if Y_consultant.shape != (
    10,
    12
):

    raise RuntimeError(
        f"Bad consultant target shape: {Y_consultant.shape}"
    )


if Y_all_clinicians.shape != (
    10,
    12
):

    raise RuntimeError(
        f"Bad all-clinician target shape: {Y_all_clinicians.shape}"
    )


if not np.allclose(
    Y_consultant.sum(axis=1),
    1.0,
    atol=1e-6
):

    raise RuntimeError(
        "Consultant targets do not sum to 1."
    )


if not np.allclose(
    Y_all_clinicians.sum(axis=1),
    1.0,
    atol=1e-6
):

    raise RuntimeError(
        "All-clinician targets do not sum to 1."
    )


print("\n" + "=" * 100)
print("FINAL GAZE TARGET SHAPES")
print("=" * 100)

print(
    "Consultant:",
    Y_consultant.shape
)

print(
    "All clinicians:",
    Y_all_clinicians.shape
)

print(
    "\n✓ Definitive equal-reader targets constructed."
)


# ======================================================================================
# 9. DISPLAY TARGETS
# ======================================================================================

consultant_df = pd.DataFrame(
    Y_consultant,
    index=ECG_ORDER,
    columns=LEAD_ORDER
)


all_clinician_df = pd.DataFrame(
    Y_all_clinicians,
    index=ECG_ORDER,
    columns=LEAD_ORDER
)


print("\n" + "=" * 100)
print("CONSULTANT TARGET — MODEL-VISIBLE 12-LEAD GAZE")
print("=" * 100)

print(
    consultant_df
    .round(4)
    .to_string()
)


print("\n" + "=" * 100)
print("ALL-CLINICIAN TARGET — MODEL-VISIBLE 12-LEAD GAZE")
print("=" * 100)

print(
    all_clinician_df
    .round(4)
    .to_string()
)


# ======================================================================================
# 10. COMPARE POPULATION TARGETS
# ======================================================================================

def cosine_similarity(a, b):

    denominator = (
        np.linalg.norm(a)
        *
        np.linalg.norm(b)
    )

    if denominator <= 0:
        raise RuntimeError(
            "Zero vector in cosine calculation."
        )

    return float(
        np.dot(a, b)
        /
        denominator
    )


comparison_rows = []


for i, ecg in enumerate(
    ECG_ORDER
):

    c = Y_consultant[i]
    a = Y_all_clinicians[i]


    comparison_rows.append(
        {
            "ECG":
                ecg,

            "cosine_similarity":
                cosine_similarity(
                    c,
                    a
                ),

            "L1_distance":
                float(
                    np.sum(
                        np.abs(
                            c - a
                        )
                    )
                ),

            "max_abs_lead_difference":
                float(
                    np.max(
                        np.abs(
                            c - a
                        )
                    )
                ),
        }
    )


comparison_df = pd.DataFrame(
    comparison_rows
)


print("\n" + "=" * 100)
print("CONSULTANT vs ALL-CLINICIAN TARGET DIFFERENCE")
print("=" * 100)

print(
    comparison_df
    .round(6)
    .to_string(
        index=False
    )
)


# ======================================================================================
# 11. SAVE TARGETS + AUDITS
# ======================================================================================

OUT_DIR = Path(
    "definitive_clean_gaze"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


TARGET_PATH = (
    OUT_DIR
    /
    "clean_gaze_targets.npz"
)


np.savez_compressed(

    TARGET_PATH,

    consultant=
        Y_consultant,

    all_clinicians=
        Y_all_clinicians,

    ecg_order=
        np.asarray(
            ECG_ORDER
        ),

    lead_order=
        np.asarray(
            LEAD_ORDER
        ),
)


# Long-format targets
long_rows = []


for ei, ecg in enumerate(
    ECG_ORDER
):

    for li, lead in enumerate(
        LEAD_ORDER
    ):

        long_rows.append(
            {
                "ECG":
                    ecg,

                "lead":
                    lead,

                "consultant":
                    float(
                        Y_consultant[
                            ei,
                            li
                        ]
                    ),

                "all_clinicians":
                    float(
                        Y_all_clinicians[
                            ei,
                            li
                        ]
                    ),
            }
        )


pd.DataFrame(
    long_rows
).to_csv(
    OUT_DIR
    /
    "clean_gaze_targets_long.csv",
    index=False
)


# Per-reader normalised gaze
clinical_norm[
    [
        "Respondent_Name",
        "_group_norm",
        "_ecg",
        "_raw_total",
        *LEAD_ORDER
    ]
].to_csv(
    OUT_DIR
    /
    "per_reader_model_visible_gaze.csv",
    index=False
)


coverage_df.to_csv(
    OUT_DIR
    /
    "model_visible_gaze_coverage.csv",
    index=False
)


comparison_df.to_csv(
    OUT_DIR
    /
    "population_target_comparison.csv",
    index=False
)


# ======================================================================================
# 12. HASH HELPER
# ======================================================================================

def sha256_file_local(path):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        while True:

            block = f.read(
                1024 * 1024
            )

            if not block:
                break

            h.update(block)

    return h.hexdigest()


TARGET_HASH = sha256_file_local(
    TARGET_PATH
)


print(
    "\nTarget SHA256:"
)

print(
    TARGET_HASH
)


# ======================================================================================
# 13. BASE MODEL AUDIT
# ======================================================================================

print("\n" + "=" * 100)
print("BASE MODEL AUDIT")
print("=" * 100)


base_model = tf.keras.models.load_model(
    BASE_MODEL_PATH,
    compile=False
)


print(
    "Model name:",
    base_model.name
)

print(
    "Input shape:",
    base_model.input_shape
)

print(
    "Output shape:",
    base_model.output_shape
)

print(
    "Number of layers:",
    len(base_model.layers)
)


conv10 = base_model.get_layer(
    "conv2d_10"
)


add3 = base_model.get_layer(
    "add_3"
)


print(
    "\nconv2d_10 output shape:",
    conv10.output.shape
)

print(
    "add_3 output shape:",
    add3.output.shape
)


final_layer = base_model.layers[
    -1
]


print(
    "\nFinal layer:",
    final_layer.name
)


if hasattr(
    final_layer,
    "activation"
):

    print(
        "Final activation:",
        final_layer.activation.__name__
    )


# ======================================================================================
# 14. FORWARD PASS — NO TRAINING
# ======================================================================================

print("\n" + "=" * 100)
print("FORWARD-PASS CHECK")
print("=" * 100)


test_output = base_model(
    X_gaze[
        0:1
    ],
    training=False
)


print(
    "Output shape:",
    tuple(
        test_output.shape
    )
)


if not np.all(
    np.isfinite(
        test_output.numpy()
    )
):

    raise RuntimeError(
        "Forward pass produced non-finite values."
    )


print(
    "✓ Base-model forward pass successful."
)


# ======================================================================================
# 15. SAVE FINAL STEP-1 MANIFEST
# ======================================================================================

manifest = {

    "created":
        datetime.now().isoformat(),

    "experiment":
        "Definitive clean gaze-aligned ECG experiment",

    "stage":
        "Step 1 complete",

    "frozen_ecg_input":
        str(
            V9_PATH
        ),

    "frozen_ecg_sha256":
        V9_HASH,

    "eye_tracking_csv":
        str(
            GAZE_CSV_PATH
        ),

    "eye_tracking_csv_sha256":
        GAZE_HASH,

    "base_model":
        str(
            BASE_MODEL_PATH
        ),

    "base_model_sha256":
        BASE_HASH,

    "clean_gaze_targets":
        str(
            TARGET_PATH
        ),

    "clean_gaze_targets_sha256":
        TARGET_HASH,

    "ecg_order":
        ECG_ORDER,

    "lead_order":
        LEAD_ORDER,

    "consultant_reader_count":
        9,

    "all_clinician_reader_count":
        20,

    "all_clinician_composition": {
        "consultant":
            9,
        "fellow":
            10,
        "resident":
            1,
    },

    "gaze_target_scope":
        (
            "Conditional distribution of dwell time within the 12 "
            "model-visible short diagnostic lead AOIs."
        ),

    "excluded_aoi_scope":
        (
            "Long rhythm-strip grid cells, Information AOIs, and "
            "non-standard additional leads not represented in the "
            "12-channel model input."
        ),

    "gaze_target_method":
        (
            "For each included clinician and ECG, dwell time within "
            "the 12 model-visible diagnostic lead AOIs was arranged "
            "in fixed standard lead order and normalised independently "
            "to sum to one. Population targets were then calculated "
            "by equally averaging the reader-level normalised vectors."
        ),

    "mean_model_visible_gaze_fraction":
        float(
            coverage_df[
                "model_visible_fraction"
            ].mean()
        ),

    "median_model_visible_gaze_fraction":
        float(
            coverage_df[
                "model_visible_fraction"
            ].median()
        ),

    "training_performed":
        False,
}


with open(
    OUT_DIR
    /
    "step1_manifest.json",
    "w"
) as f:

    json.dump(
        manifest,
        f,
        indent=2
    )


# ======================================================================================
# 16. FINISH
# ======================================================================================

print("\n\n" + "=" * 100)
print("STEP 1 COMPLETE — DEFINITIVE TARGETS SAVED — NO TRAINING")
print("=" * 100)

print(
    """
Please paste back:

1. CLINICAL POPULATION COMPLETENESS
2. PER-READER MODEL-VISIBLE GAZE NORMALISATION
3. CONSULTANT vs ALL-CLINICIAN TARGET DIFFERENCE
4. BASE MODEL AUDIT
5. FORWARD-PASS CHECK

If those pass, the next step is the prespecified 5-fold
held-out-ECG gaze-alignment experiment.
"""
)

In [ ]:
# ======================================================================================
# STEP 2A — FREEZE THE LEAD-ATTENTION READOUT
#
# NO TRAINING.
#
# We inspect:
#   1. architecture immediately around conv2d_10 / add_3
#   2. whether add_3 activations are non-negative
#   3. candidate 12-lead readouts on the frozen 10 ECGs
# ======================================================================================

import numpy as np
import pandas as pd
import tensorflow as tf


# ======================================================================================
# 1. SHOW MODEL REGION AROUND conv2d_10 AND add_3
# ======================================================================================

layer_names = [
    layer.name
    for layer in base_model.layers
]

conv_idx = layer_names.index(
    "conv2d_10"
)

add_idx = layer_names.index(
    "add_3"
)


start = max(
    0,
    min(conv_idx, add_idx) - 5
)

end = min(
    len(base_model.layers),
    max(conv_idx, add_idx) + 6
)


print("=" * 100)
print("ARCHITECTURE AROUND conv2d_10 / add_3")
print("=" * 100)


for i in range(start, end):

    layer = base_model.layers[i]

    activation = ""

    if hasattr(layer, "activation"):

        try:
            activation = (
                " | activation="
                + layer.activation.__name__
            )

        except Exception:
            pass

    print(
        f"{i:02d} | "
        f"{layer.name:25s} | "
        f"{layer.__class__.__name__:25s} | "
        f"output={layer.output.shape}"
        f"{activation}"
    )


# ======================================================================================
# 2. BUILD REPRESENTATION MODEL
# ======================================================================================

representation_model = tf.keras.Model(
    inputs=base_model.input,
    outputs=base_model.get_layer(
        "add_3"
    ).output
)


A = representation_model(
    X_gaze,
    training=False
).numpy()


print("\n" + "=" * 100)
print("add_3 ACTIVATION AUDIT")
print("=" * 100)

print(
    "Shape:",
    A.shape
)

print(
    "Finite:",
    np.all(
        np.isfinite(A)
    )
)

print(
    "Global minimum:",
    float(
        np.min(A)
    )
)

print(
    "Global maximum:",
    float(
        np.max(A)
    )
)

print(
    "Global mean:",
    float(
        np.mean(A)
    )
)

print(
    "Fraction negative:",
    float(
        np.mean(
            A < 0
        )
    )
)

print(
    "Fraction exactly zero:",
    float(
        np.mean(
            A == 0
        )
    )
)


# ======================================================================================
# 3. PER-ECG ACTIVATION RANGE
# ======================================================================================

activation_rows = []


for i, ecg in enumerate(
    ECG_ORDER
):

    x = A[i]

    activation_rows.append(
        {
            "ECG":
                ecg,

            "min":
                float(
                    np.min(x)
                ),

            "max":
                float(
                    np.max(x)
                ),

            "mean":
                float(
                    np.mean(x)
                ),

            "fraction_negative":
                float(
                    np.mean(
                        x < 0
                    )
                ),

            "fraction_zero":
                float(
                    np.mean(
                        x == 0
                    )
                ),
        }
    )


activation_df = pd.DataFrame(
    activation_rows
)


print("\n" + "=" * 100)
print("PER-ECG add_3 RANGE")
print("=" * 100)

print(
    activation_df
    .round(6)
    .to_string(
        index=False
    )
)


# ======================================================================================
# 4. CANDIDATE LEAD READOUTS
#
# These are diagnostic only.
#
# A. signed mean:
#       mean activation over time × channels
#
# B. mean absolute activation:
#       mean |activation| over time × channels
#
# C. RMS energy:
#       sqrt(mean(activation^2))
# ======================================================================================

signed_mean = np.mean(
    A,
    axis=(
        2,
        3
    )
)


abs_mean = np.mean(
    np.abs(A),
    axis=(
        2,
        3
    )
)


rms = np.sqrt(
    np.mean(
        np.square(A),
        axis=(
            2,
            3
        )
    )
)


print("\n" + "=" * 100)
print("CANDIDATE READOUT SHAPES")
print("=" * 100)

print(
    "signed mean:",
    signed_mean.shape
)

print(
    "absolute mean:",
    abs_mean.shape
)

print(
    "RMS:",
    rms.shape
)


print("\nSigned-mean minimum:")
print(
    float(
        signed_mean.min()
    )
)


print(
    "Absolute-mean minimum:",
    float(
        abs_mean.min()
    )
)


print(
    "RMS minimum:",
    float(
        rms.min()
    )
)


# ======================================================================================
# 5. CHECK WHETHER SIMPLE NORMALISATION OF SIGNED MEAN IS VALID
# ======================================================================================

signed_valid = bool(
    np.all(
        signed_mean >= 0
    )
    and
    np.all(
        signed_mean.sum(
            axis=1
        )
        >
        0
    )
)


print("\n" + "=" * 100)
print("READOUT DECISION INPUT")
print("=" * 100)

print(
    "Is signed mean non-negative for every ECG/lead?",
    signed_valid
)


# ======================================================================================
# 6. SHOW NORMALISED ABS-MEAN AND RMS DISTRIBUTIONS
# ======================================================================================

abs_dist = (
    abs_mean
    /
    abs_mean.sum(
        axis=1,
        keepdims=True
    )
)


rms_dist = (
    rms
    /
    rms.sum(
        axis=1,
        keepdims=True
    )
)


abs_df = pd.DataFrame(
    abs_dist,
    index=ECG_ORDER,
    columns=LEAD_ORDER
)


rms_df = pd.DataFrame(
    rms_dist,
    index=ECG_ORDER,
    columns=LEAD_ORDER
)


print("\n" + "=" * 100)
print("BASE MODEL — NORMALISED MEAN-ABSOLUTE add_3 READOUT")
print("=" * 100)

print(
    abs_df
    .round(4)
    .to_string()
)


print("\n" + "=" * 100)
print("BASE MODEL — NORMALISED RMS add_3 READOUT")
print("=" * 100)

print(
    rms_df
    .round(4)
    .to_string()
)


print(
    """
NO TRAINING HAS OCCURRED.

Please paste back:

1. ARCHITECTURE AROUND conv2d_10 / add_3
2. add_3 ACTIVATION AUDIT
3. PER-ECG add_3 RANGE
4. READOUT DECISION INPUT

Then we will freeze ONE lead-attention definition and use that
unchanged for every fold, both gaze populations, and all evaluations.
"""
)

In [ ]:
# ======================================================================================
# DEFINITIVE CLEAN GAZE EXPERIMENT
# STEP 2 — 5-FOLD HELD-OUT-ECG GAZE ALIGNMENT
#
# FROZEN:
#   ECG input: final_digitisation_v9/X_gaze_final.npy
#   Targets: definitive_clean_gaze/clean_gaze_targets.npz
#
# Attention:
#   add_3 -> RMS over (time, channels) -> normalise across 12 leads
#
# Training:
#   5 folds, 2 held-out ECGs/fold
#   only conv2d_10 trainable
#   BatchNorm frozen
#   training=False throughout forward passes
#   gaze-only MSE
#   Adam 1e-4
#   25 epochs
#   40 noisy copies/training ECG
#   Gaussian noise SD = 0.03
#   batch size = 16
#
# Consultant and all-clinician arms use identical:
#   folds
#   starting weights
#   augmented X
#   batch order
#
# Only the gaze target differs.
# ======================================================================================

import gc
import json
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import KFold
from scipy.stats import wilcoxon


# ======================================================================================
# 1. PARAMETERS
# ======================================================================================

SEED = 2026
N_SPLITS = 5

AUGMENTATIONS_PER_ECG = 40
NOISE_SD = 0.03

BATCH_SIZE = 16
EPOCHS = 25

LEARNING_RATE = 1e-4
RMS_EPSILON = 1e-8


ECG_ORDER = [
    "NSR",
    "AFib",
    "Hyperkalemia",
    "AFlutter",
    "VTach",
    "WPW",
    "VentPaced",
    "LBBB",
    "STEMI",
    "CompleteHeartBlock",
]


LEAD_ORDER = [
    "I",
    "II",
    "III",
    "aVR",
    "aVL",
    "aVF",
    "V1",
    "V2",
    "V3",
    "V4",
    "V5",
    "V6",
]


POPULATIONS = [
    "consultant",
    "all_clinicians",
]


# ======================================================================================
# 2. REPRODUCIBILITY
# ======================================================================================

np.random.seed(SEED)
tf.random.set_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
    print("TensorFlow deterministic operations enabled.")
except Exception as e:
    print("Could not explicitly enable deterministic operations:", repr(e))


# ======================================================================================
# 3. PATHS
# ======================================================================================

ROOT = Path("definitive_clean_gaze")

STEP1_MANIFEST = ROOT / "step1_manifest.json"
TARGET_PATH = ROOT / "clean_gaze_targets.npz"

X_PATH = Path(
    "final_digitisation_v9/X_gaze_final.npy"
)

STEP2_DIR = ROOT / "step2_cv"
STEP2_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MODEL_DIR = STEP2_DIR / "fold_weights"
MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)


for p in [
    STEP1_MANIFEST,
    TARGET_PATH,
    X_PATH,
]:
    if not p.exists():
        raise FileNotFoundError(p)


# ======================================================================================
# 4. HASH HELPER
# ======================================================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            block = f.read(
                1024 * 1024
            )

            if not block:
                break

            h.update(block)

    return h.hexdigest()


# ======================================================================================
# 5. VERIFY FROZEN FILES
# ======================================================================================

with open(
    STEP1_MANIFEST,
    "r"
) as f:

    step1_manifest = json.load(f)


BASE_MODEL_PATH = Path(
    step1_manifest["base_model"]
)


if not BASE_MODEL_PATH.exists():
    raise FileNotFoundError(
        BASE_MODEL_PATH
    )


current_x_hash = sha256_file(
    X_PATH
)

current_target_hash = sha256_file(
    TARGET_PATH
)

current_base_hash = sha256_file(
    BASE_MODEL_PATH
)


print("=" * 100)
print("FROZEN INPUT VERIFICATION")
print("=" * 100)

print("V9 ECG hash:")
print(current_x_hash)

print("\nTarget hash:")
print(current_target_hash)

print("\nBase-model hash:")
print(current_base_hash)


if (
    current_x_hash
    !=
    step1_manifest["frozen_ecg_sha256"]
):
    raise RuntimeError(
        "Frozen V9 ECG input hash no longer matches Step 1."
    )


if (
    current_target_hash
    !=
    step1_manifest["clean_gaze_targets_sha256"]
):
    raise RuntimeError(
        "Gaze-target hash no longer matches Step 1."
    )


if (
    current_base_hash
    !=
    step1_manifest["base_model_sha256"]
):
    raise RuntimeError(
        "Base-model hash no longer matches Step 1."
    )


print(
    "\n✓ All frozen hashes match Step 1."
)


# ======================================================================================
# 6. LOAD DATA
# ======================================================================================

X = np.load(
    X_PATH
).astype(
    np.float32
)


target_data = np.load(
    TARGET_PATH
)


Y = {

    "consultant":
        target_data[
            "consultant"
        ].astype(
            np.float32
        ),

    "all_clinicians":
        target_data[
            "all_clinicians"
        ].astype(
            np.float32
        ),
}


saved_ecg_order = [
    str(x)
    for x in target_data[
        "ecg_order"
    ]
]


saved_lead_order = [
    str(x)
    for x in target_data[
        "lead_order"
    ]
]


if saved_ecg_order != ECG_ORDER:
    raise RuntimeError(
        "Saved ECG order mismatch."
    )


if saved_lead_order != LEAD_ORDER:
    raise RuntimeError(
        "Saved lead order mismatch."
    )


if X.shape != (
    10,
    12,
    1000,
    1
):
    raise RuntimeError(
        f"Unexpected X shape: {X.shape}"
    )


for population in POPULATIONS:

    if Y[population].shape != (
        10,
        12
    ):
        raise RuntimeError(
            f"Unexpected target shape for {population}: "
            f"{Y[population].shape}"
        )


print(
    "\n✓ Frozen arrays loaded."
)


# ======================================================================================
# 7. FROZEN ATTENTION READOUT
# ======================================================================================

def attention_distribution_from_add3(
    add3_tensor
):

    # add_3:
    # (batch, 12, 55, 128)

    rms = tf.sqrt(
        tf.reduce_mean(
            tf.square(
                add3_tensor
            ),
            axis=[
                2,
                3
            ]
        )
        +
        RMS_EPSILON
    )


    denominator = tf.reduce_sum(
        rms,
        axis=1,
        keepdims=True
    )


    return (
        rms
        /
        (
            denominator
            +
            RMS_EPSILON
        )
    )


# ======================================================================================
# 8. METRICS
# ======================================================================================

def cosine_metric(p, q):

    p = np.asarray(
        p,
        dtype=np.float64
    )

    q = np.asarray(
        q,
        dtype=np.float64
    )


    denominator = (
        np.linalg.norm(p)
        *
        np.linalg.norm(q)
    )


    if denominator <= 0:
        return np.nan


    return float(
        np.dot(p, q)
        /
        denominator
    )


def mse_metric(p, q):

    p = np.asarray(
        p,
        dtype=np.float64
    )

    q = np.asarray(
        q,
        dtype=np.float64
    )


    return float(
        np.mean(
            np.square(
                p - q
            )
        )
    )


def jsd_metric(
    p,
    q,
    eps=1e-12
):

    p = np.asarray(
        p,
        dtype=np.float64
    )

    q = np.asarray(
        q,
        dtype=np.float64
    )


    p = np.clip(
        p,
        eps,
        None
    )

    q = np.clip(
        q,
        eps,
        None
    )


    p /= p.sum()
    q /= q.sum()


    m = 0.5 * (
        p + q
    )


    kl_pm = np.sum(
        p
        *
        np.log(
            p / m
        )
    )


    kl_qm = np.sum(
        q
        *
        np.log(
            q / m
        )
    )


    return float(
        0.5
        *
        (
            kl_pm
            +
            kl_qm
        )
    )


# ======================================================================================
# 9. LOAD A FRESH MODEL FOR EACH ARM/FOLD
# ======================================================================================

def load_fresh_representation_model():

    tf.keras.backend.clear_session()

    tf.random.set_seed(
        SEED
    )


    model = tf.keras.models.load_model(
        BASE_MODEL_PATH,
        compile=False
    )


    # Freeze entire network.
    for layer in model.layers:
        layer.trainable = False


    # Only this layer may change.
    conv10 = model.get_layer(
        "conv2d_10"
    )

    conv10.trainable = True


    add3 = model.get_layer(
        "add_3"
    )


    rep_model = tf.keras.Model(
        inputs=model.input,
        outputs=add3.output
    )


    train_vars = list(
        conv10.trainable_variables
    )


    if len(train_vars) == 0:
        raise RuntimeError(
            "conv2d_10 has no trainable variables."
        )


    return (
        model,
        rep_model,
        conv10,
        train_vars
    )


# ======================================================================================
# 10. PREDICT LEAD ATTENTION
# ======================================================================================

def predict_attention(
    rep_model,
    x
):

    activations = rep_model(
        x,
        training=False
    )


    attention = (
        attention_distribution_from_add3(
            activations
        )
        .numpy()
    )


    if not np.all(
        np.isfinite(
            attention
        )
    ):
        raise RuntimeError(
            "Non-finite attention values."
        )


    if not np.allclose(
        attention.sum(
            axis=1
        ),
        1.0,
        atol=1e-5
    ):
        raise RuntimeError(
            "Attention distributions do not sum to 1."
        )


    if np.any(
        attention < 0
    ):
        raise RuntimeError(
            "Negative attention detected."
        )


    return attention


# ======================================================================================
# 11. SNAPSHOT / COMPARE WEIGHTS
# ======================================================================================

def snapshot_layer_weights(
    model
):

    snapshot = {}


    for layer in model.layers:

        snapshot[
            layer.name
        ] = [
            np.array(
                weight,
                copy=True
            )
            for weight in layer.get_weights()
        ]


    return snapshot


def compare_layer_weights(
    before,
    model
):

    rows = []


    for layer in model.layers:

        old_weights = before[
            layer.name
        ]

        new_weights = layer.get_weights()


        if len(old_weights) != len(new_weights):
            raise RuntimeError(
                f"Weight-count mismatch: {layer.name}"
            )


        max_difference = 0.0


        for old, new in zip(
            old_weights,
            new_weights
        ):

            if old.shape != new.shape:
                raise RuntimeError(
                    f"Weight-shape mismatch: {layer.name}"
                )


            if old.size > 0:

                difference = float(
                    np.max(
                        np.abs(
                            old - new
                        )
                    )
                )


                max_difference = max(
                    max_difference,
                    difference
                )


        rows.append(
            {
                "layer":
                    layer.name,

                "n_weight_tensors":
                    len(
                        new_weights
                    ),

                "max_abs_change":
                    max_difference,
            }
        )


    return pd.DataFrame(
        rows
    )


# ======================================================================================
# 12. FREEZE CROSS-VALIDATION SPLITS
# ======================================================================================

kf = KFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED
)


folds = []


for fold_number, split in enumerate(
    kf.split(
        np.arange(
            len(
                ECG_ORDER
            )
        )
    ),
    start=1
):

    train_indices, test_indices = split


    folds.append(
        {
            "fold":
                fold_number,

            "train_indices":
                np.asarray(
                    train_indices,
                    dtype=int
                ),

            "test_indices":
                np.asarray(
                    test_indices,
                    dtype=int
                ),
        }
    )


print(
    "\n" +
    "=" * 100
)

print(
    "FROZEN 5-FOLD SPLIT"
)

print(
    "=" * 100
)


fold_manifest_rows = []


for fold in folds:

    train_names = [
        ECG_ORDER[i]
        for i in fold[
            "train_indices"
        ]
    ]


    test_names = [
        ECG_ORDER[i]
        for i in fold[
            "test_indices"
        ]
    ]


    print(
        f"\nFold {fold['fold']}"
    )

    print(
        "  held out:",
        test_names
    )

    print(
        "  training:",
        train_names
    )


    fold_manifest_rows.append(
        {
            "fold":
                fold[
                    "fold"
                ],

            "held_out_ECGs":
                " | ".join(
                    test_names
                ),

            "training_ECGs":
                " | ".join(
                    train_names
                ),
        }
    )


pd.DataFrame(
    fold_manifest_rows
).to_csv(
    STEP2_DIR
    /
    "fold_manifest.csv",
    index=False
)


# ======================================================================================
# 13. TRAIN ONE POPULATION × FOLD
# ======================================================================================

def train_one_arm(
    population,
    fold_number,
    train_indices,
    test_indices,
    X_train_augmented,
    epoch_permutations
):

    (
        model,
        rep_model,
        conv10,
        train_vars
    ) = load_fresh_representation_model()


    before_weights = snapshot_layer_weights(
        model
    )


    print(
        "\nTrainable conv2d_10 variables:"
    )

    for variable in train_vars:

        print(
            "   ",
            variable.name,
            tuple(
                variable.shape
            )
        )


    # ------------------------------------------------------------------
    # Baseline held-out attention
    # ------------------------------------------------------------------

    X_test = X[
        test_indices
    ]


    Y_test = Y[
        population
    ][
        test_indices
    ]


    baseline_attention = predict_attention(
        rep_model,
        X_test
    )


    # ------------------------------------------------------------------
    # Training targets
    # ------------------------------------------------------------------

    Y_train_augmented = np.repeat(
        Y[
            population
        ][
            train_indices
        ],
        AUGMENTATIONS_PER_ECG,
        axis=0
    ).astype(
        np.float32
    )


    if (
        len(
            X_train_augmented
        )
        !=
        len(
            Y_train_augmented
        )
    ):
        raise RuntimeError(
            "Augmented X/Y size mismatch."
        )


    optimizer = tf.keras.optimizers.Adam(
        learning_rate=LEARNING_RATE
    )


    epoch_losses = []


    # ------------------------------------------------------------------
    # Deterministic custom loop
    # ------------------------------------------------------------------

    for epoch in range(
        EPOCHS
    ):

        permutation = epoch_permutations[
            epoch
        ]


        batch_losses = []


        for start in range(
            0,
            len(
                permutation
            ),
            BATCH_SIZE
        ):

            batch_idx = permutation[
                start:
                start + BATCH_SIZE
            ]


            xb = tf.convert_to_tensor(
                X_train_augmented[
                    batch_idx
                ],
                dtype=tf.float32
            )


            yb = tf.convert_to_tensor(
                Y_train_augmented[
                    batch_idx
                ],
                dtype=tf.float32
            )


            with tf.GradientTape() as tape:

                add3 = rep_model(
                    xb,
                    training=False
                )


                predicted_attention = (
                    attention_distribution_from_add3(
                        add3
                    )
                )


                loss = tf.reduce_mean(
                    tf.square(
                        predicted_attention
                        -
                        yb
                    )
                )


            gradients = tape.gradient(
                loss,
                train_vars
            )


            if any(
                gradient is None
                for gradient in gradients
            ):
                raise RuntimeError(
                    "At least one conv2d_10 gradient is None."
                )


            for gradient in gradients:

                if not np.all(
                    np.isfinite(
                        gradient.numpy()
                    )
                ):
                    raise RuntimeError(
                        "Non-finite gradient detected."
                    )


            optimizer.apply_gradients(
                zip(
                    gradients,
                    train_vars
                )
            )


            batch_losses.append(
                float(
                    loss.numpy()
                )
            )


        mean_epoch_loss = float(
            np.mean(
                batch_losses
            )
        )


        epoch_losses.append(
            mean_epoch_loss
        )


        print(
            f"    epoch {epoch + 1:02d}/{EPOCHS}: "
            f"loss={mean_epoch_loss:.8f}"
        )


    # ------------------------------------------------------------------
    # Held-out attention after training
    # ------------------------------------------------------------------

    trained_attention = predict_attention(
        rep_model,
        X_test
    )


    # ------------------------------------------------------------------
    # Confirm only conv2d_10 changed
    # ------------------------------------------------------------------

    weight_df = compare_layer_weights(
        before_weights,
        model
    )


    changed_layers = set(
        weight_df.loc[
            weight_df[
                "max_abs_change"
            ]
            >
            1e-12,
            "layer"
        ]
    )


    if changed_layers != {
        "conv2d_10"
    }:
        raise RuntimeError(
            "\nUnexpected changed layers.\n"
            "Expected: {'conv2d_10'}\n"
            f"Actual:   {changed_layers}"
        )


    print(
        "\n    ✓ Weight audit: ONLY conv2d_10 changed."
    )


    # ------------------------------------------------------------------
    # Save weights
    # ------------------------------------------------------------------

    weight_path = (
        MODEL_DIR
        /
        f"{population}_fold{fold_number}.weights.h5"
    )


    model.save_weights(
        weight_path
    )


    return {
        "baseline_attention":
            baseline_attention,

        "trained_attention":
            trained_attention,

        "Y_test":
            Y_test,

        "epoch_losses":
            epoch_losses,

        "weight_df":
            weight_df,

        "weight_path":
            weight_path,
    }


# ======================================================================================
# 14. RUN FIVE-FOLD EXPERIMENT
# ======================================================================================

metric_rows = []
prediction_rows = []
history_rows = []
weight_audit_rows = []


for fold in folds:

    fold_number = fold[
        "fold"
    ]


    train_indices = fold[
        "train_indices"
    ]


    test_indices = fold[
        "test_indices"
    ]


    print(
        "\n\n" +
        "#" * 100
    )

    print(
        f"FOLD {fold_number}"
    )

    print(
        "#" * 100
    )


    print(
        "Held-out ECGs:",
        [
            ECG_ORDER[i]
            for i in test_indices
        ]
    )


    # ------------------------------------------------------------------
    # Generate augmented X exactly once per fold
    # ------------------------------------------------------------------

    augmentation_rng = np.random.default_rng(
        SEED
        +
        10000
        *
        fold_number
    )


    X_train_base = np.repeat(
        X[
            train_indices
        ],
        AUGMENTATIONS_PER_ECG,
        axis=0
    ).astype(
        np.float32
    )


    noise = augmentation_rng.normal(
        loc=0.0,
        scale=NOISE_SD,
        size=X_train_base.shape
    ).astype(
        np.float32
    )


    X_train_augmented = (
        X_train_base
        +
        noise
    ).astype(
        np.float32
    )


    expected_training_samples = (
        len(
            train_indices
        )
        *
        AUGMENTATIONS_PER_ECG
    )


    if (
        X_train_augmented.shape[0]
        !=
        expected_training_samples
    ):
        raise RuntimeError(
            "Unexpected augmented training-set size."
        )


    # ------------------------------------------------------------------
    # Same epoch permutations for both populations
    # ------------------------------------------------------------------

    permutation_rng = np.random.default_rng(
        SEED
        +
        20000
        *
        fold_number
    )


    epoch_permutations = [
        permutation_rng.permutation(
            expected_training_samples
        )
        for _ in range(
            EPOCHS
        )
    ]


    # ------------------------------------------------------------------
    # Both populations
    # ------------------------------------------------------------------

    for population in POPULATIONS:

        print(
            "\n" +
            "=" * 100
        )

        print(
            f"FOLD {fold_number} — {population.upper()}"
        )

        print(
            "=" * 100
        )


        result = train_one_arm(
            population=population,
            fold_number=fold_number,
            train_indices=train_indices,
            test_indices=test_indices,
            X_train_augmented=X_train_augmented,
            epoch_permutations=epoch_permutations,
        )


        baseline_attention = result[
            "baseline_attention"
        ]


        trained_attention = result[
            "trained_attention"
        ]


        Y_test = result[
            "Y_test"
        ]


        # --------------------------------------------------------------
        # Training history
        # --------------------------------------------------------------

        for epoch_idx, loss_value in enumerate(
            result[
                "epoch_losses"
            ],
            start=1
        ):

            history_rows.append(
                {
                    "population":
                        population,

                    "fold":
                        fold_number,

                    "epoch":
                        epoch_idx,

                    "loss":
                        loss_value,
                }
            )


        # --------------------------------------------------------------
        # Weight audit
        # --------------------------------------------------------------

        temporary_weight_df = result[
            "weight_df"
        ].copy()


        temporary_weight_df[
            "population"
        ] = population


        temporary_weight_df[
            "fold"
        ] = fold_number


        weight_audit_rows.extend(
            temporary_weight_df.to_dict(
                "records"
            )
        )


        # --------------------------------------------------------------
        # Held-out ECGs
        # --------------------------------------------------------------

        for local_idx, ecg_index in enumerate(
            test_indices
        ):

            ecg = ECG_ORDER[
                ecg_index
            ]


            target = Y_test[
                local_idx
            ]


            base_pred = baseline_attention[
                local_idx
            ]


            trained_pred = trained_attention[
                local_idx
            ]


            base_cos = cosine_metric(
                base_pred,
                target
            )


            trained_cos = cosine_metric(
                trained_pred,
                target
            )


            base_mse = mse_metric(
                base_pred,
                target
            )


            trained_mse = mse_metric(
                trained_pred,
                target
            )


            base_jsd = jsd_metric(
                base_pred,
                target
            )


            trained_jsd = jsd_metric(
                trained_pred,
                target
            )


            metric_rows.append(
                {
                    "population":
                        population,

                    "fold":
                        fold_number,

                    "ECG":
                        ecg,

                    "baseline_cosine":
                        base_cos,

                    "trained_cosine":
                        trained_cos,

                    "delta_cosine":
                        trained_cos
                        -
                        base_cos,

                    "baseline_MSE":
                        base_mse,

                    "trained_MSE":
                        trained_mse,

                    "delta_MSE":
                        trained_mse
                        -
                        base_mse,

                    "baseline_JSD":
                        base_jsd,

                    "trained_JSD":
                        trained_jsd,

                    "delta_JSD":
                        trained_jsd
                        -
                        base_jsd,
                }
            )


            for lead_idx, lead in enumerate(
                LEAD_ORDER
            ):

                prediction_rows.append(
                    {
                        "population":
                            population,

                        "fold":
                            fold_number,

                        "ECG":
                            ecg,

                        "lead":
                            lead,

                        "target":
                            float(
                                target[
                                    lead_idx
                                ]
                            ),

                        "baseline":
                            float(
                                base_pred[
                                    lead_idx
                                ]
                            ),

                        "trained":
                            float(
                                trained_pred[
                                    lead_idx
                                ]
                            ),
                    }
                )


        gc.collect()


# ======================================================================================
# 15. CREATE / SAVE RAW OUTPUT TABLES
# ======================================================================================

metrics_df = pd.DataFrame(
    metric_rows
)


predictions_df = pd.DataFrame(
    prediction_rows
)


history_df = pd.DataFrame(
    history_rows
)


weight_audit_df = pd.DataFrame(
    weight_audit_rows
)


metrics_df.to_csv(
    STEP2_DIR
    /
    "heldout_metrics.csv",
    index=False
)


predictions_df.to_csv(
    STEP2_DIR
    /
    "heldout_attention_distributions.csv",
    index=False
)


history_df.to_csv(
    STEP2_DIR
    /
    "training_history.csv",
    index=False
)


weight_audit_df.to_csv(
    STEP2_DIR
    /
    "weight_change_audit.csv",
    index=False
)


# ======================================================================================
# 16. VERIFY HELD-OUT COVERAGE
# ======================================================================================

for population in POPULATIONS:

    temp = metrics_df[
        metrics_df[
            "population"
        ]
        ==
        population
    ]


    counts = temp[
        "ECG"
    ].value_counts()


    if len(counts) != 10:
        raise RuntimeError(
            f"{population}: not all 10 ECGs were evaluated."
        )


    if not (
        counts == 1
    ).all():
        raise RuntimeError(
            f"{population}: ECG holdout count error."
        )


print(
    "\n✓ Every ECG was held out exactly once for each population."
)


# ======================================================================================
# 17. WILCOXON
# ======================================================================================

def safe_wilcoxon(
    baseline,
    trained
):

    baseline = np.asarray(
        baseline,
        dtype=np.float64
    )


    trained = np.asarray(
        trained,
        dtype=np.float64
    )


    differences = (
        trained
        -
        baseline
    )


    if np.allclose(
        differences,
        0
    ):
        return (
            0.0,
            1.0
        )


    result = wilcoxon(
        trained,
        baseline,
        alternative="two-sided",
        zero_method="wilcox",
        method="auto"
    )


    return (
        float(
            result.statistic
        ),
        float(
            result.pvalue
        )
    )


# ======================================================================================
# 18. SUMMARY STATISTICS
# ======================================================================================

summary_rows = []


for population in POPULATIONS:

    temp = metrics_df[
        metrics_df[
            "population"
        ]
        ==
        population
    ].copy()


    # ------------------------------------------------------------------
    # COSINE — higher is better
    # ------------------------------------------------------------------

    statistic, p_value = safe_wilcoxon(
        temp[
            "baseline_cosine"
        ],
        temp[
            "trained_cosine"
        ]
    )


    summary_rows.append(
        {
            "population":
                population,

            "metric":
                "cosine",

            "direction_better":
                "higher",

            "baseline_mean":
                temp[
                    "baseline_cosine"
                ].mean(),

            "trained_mean":
                temp[
                    "trained_cosine"
                ].mean(),

            "mean_delta":
                temp[
                    "delta_cosine"
                ].mean(),

            "median_delta":
                temp[
                    "delta_cosine"
                ].median(),

            "n_improved":
                int(
                    (
                        temp[
                            "delta_cosine"
                        ]
                        >
                        0
                    ).sum()
                ),

            "wilcoxon_statistic":
                statistic,

            "wilcoxon_p":
                p_value,
        }
    )


    # ------------------------------------------------------------------
    # MSE — lower is better
    # ------------------------------------------------------------------

    statistic, p_value = safe_wilcoxon(
        temp[
            "baseline_MSE"
        ],
        temp[
            "trained_MSE"
        ]
    )


    summary_rows.append(
        {
            "population":
                population,

            "metric":
                "MSE",

            "direction_better":
                "lower",

            "baseline_mean":
                temp[
                    "baseline_MSE"
                ].mean(),

            "trained_mean":
                temp[
                    "trained_MSE"
                ].mean(),

            "mean_delta":
                temp[
                    "delta_MSE"
                ].mean(),

            "median_delta":
                temp[
                    "delta_MSE"
                ].median(),

            "n_improved":
                int(
                    (
                        temp[
                            "delta_MSE"
                        ]
                        <
                        0
                    ).sum()
                ),

            "wilcoxon_statistic":
                statistic,

            "wilcoxon_p":
                p_value,
        }
    )


    # ------------------------------------------------------------------
    # JSD — lower is better
    # ------------------------------------------------------------------

    statistic, p_value = safe_wilcoxon(
        temp[
            "baseline_JSD"
        ],
        temp[
            "trained_JSD"
        ]
    )


    summary_rows.append(
        {
            "population":
                population,

            "metric":
                "JSD",

            "direction_better":
                "lower",

            "baseline_mean":
                temp[
                    "baseline_JSD"
                ].mean(),

            "trained_mean":
                temp[
                    "trained_JSD"
                ].mean(),

            "mean_delta":
                temp[
                    "delta_JSD"
                ].mean(),

            "median_delta":
                temp[
                    "delta_JSD"
                ].median(),

            "n_improved":
                int(
                    (
                        temp[
                            "delta_JSD"
                        ]
                        <
                        0
                    ).sum()
                ),

            "wilcoxon_statistic":
                statistic,

            "wilcoxon_p":
                p_value,
        }
    )


summary_df = pd.DataFrame(
    summary_rows
)


summary_df.to_csv(
    STEP2_DIR
    /
    "alignment_summary.csv",
    index=False
)


# ======================================================================================
# 19. PRINT HELD-OUT RESULTS
# ======================================================================================

print(
    "\n\n" +
    "=" * 100
)

print(
    "HELD-OUT ECG RESULTS"
)

print(
    "=" * 100
)


print(
    metrics_df[
        [
            "population",
            "fold",
            "ECG",
            "baseline_cosine",
            "trained_cosine",
            "delta_cosine",
            "baseline_MSE",
            "trained_MSE",
            "delta_MSE",
            "baseline_JSD",
            "trained_JSD",
            "delta_JSD",
        ]
    ]
    .round(
        7
    )
    .to_string(
        index=False
    )
)


# ======================================================================================
# 20. PRINT PRIMARY SUMMARY
# ======================================================================================

print(
    "\n\n" +
    "=" * 100
)

print(
    "DEFINITIVE HELD-OUT GAZE-ALIGNMENT SUMMARY"
)

print(
    "=" * 100
)


print(
    summary_df
    .round(
        7
    )
    .to_string(
        index=False
    )
)


# ======================================================================================
# 21. FINAL WEIGHT AUDIT
# ======================================================================================

changed_summary = (
    weight_audit_df[
        weight_audit_df[
            "max_abs_change"
        ]
        >
        1e-12
    ][
        [
            "population",
            "fold",
            "layer",
            "max_abs_change",
        ]
    ]
)


print(
    "\n\n" +
    "=" * 100
)

print(
    "LAYERS THAT CHANGED"
)

print(
    "=" * 100
)


print(
    changed_summary.to_string(
        index=False
    )
)


if set(
    changed_summary[
        "layer"
    ].unique()
) != {
    "conv2d_10"
}:
    raise RuntimeError(
        "Final weight audit failed."
    )


if len(
    changed_summary
) != (
    N_SPLITS
    *
    len(
        POPULATIONS
    )
):
    raise RuntimeError(
        "Expected exactly one changed layer per population × fold."
    )


print(
    "\n✓ Every fold/arm changed ONLY conv2d_10."
)


# ======================================================================================
# 22. SAVE STEP-2 MANIFEST
# ======================================================================================

step2_manifest = {

    "experiment":
        "Definitive clean gaze-aligned ECG experiment",

    "stage":
        "Step 2: five-fold held-out ECG gaze alignment",

    "seed":
        SEED,

    "n_folds":
        N_SPLITS,

    "held_out_ECGs_per_fold":
        2,

    "training_ECGs_per_fold":
        8,

    "attention_readout":
        (
            "RMS activation energy from add_3: "
            "sqrt(mean(add_3^2)) across temporal and channel axes, "
            "then normalised across 12 leads."
        ),

    "attention_layer":
        "add_3",

    "trainable_layer":
        "conv2d_10",

    "batchnorm_mode":
        "frozen",

    "forward_training_mode":
        False,

    "loss":
        "MSE between 12-lead RMS attention distribution and gaze target",

    "optimizer":
        "Adam",

    "learning_rate":
        LEARNING_RATE,

    "epochs":
        EPOCHS,

    "augmentations_per_training_ECG":
        AUGMENTATIONS_PER_ECG,

    "gaussian_noise_sigma":
        NOISE_SD,

    "batch_size":
        BATCH_SIZE,

    "same_folds_between_populations":
        True,

    "same_noisy_training_examples_between_populations":
        True,

    "same_batch_order_between_populations":
        True,

    "populations": {
        "consultant":
            9,

        "all_clinicians":
            20,
    },

    "frozen_V9_sha256":
        current_x_hash,

    "frozen_gaze_target_sha256":
        current_target_hash,

    "base_model_sha256":
        current_base_hash,

    "statistical_unit":
        "held-out ECG",

    "n_statistical_units_per_population":
        10,

    "statistical_test":
        "two-sided paired Wilcoxon signed-rank test",

    "metrics": [
        "cosine similarity",
        "mean squared error",
        "Jensen-Shannon divergence",
    ],
}


with open(
    STEP2_DIR
    /
    "step2_manifest.json",
    "w"
) as f:

    json.dump(
        step2_manifest,
        f,
        indent=2
    )


print(
    "\n\n" +
    "=" * 100
)

print(
    "STEP 2 COMPLETE"
)

print(
    "=" * 100
)


print(
    """
Please paste back:

1. FROZEN 5-FOLD SPLIT

2. DEFINITIVE HELD-OUT GAZE-ALIGNMENT SUMMARY

3. LAYERS THAT CHANGED

You do not need to paste all epoch losses unless there is an error.
"""
)